# FRCRN_SE

In [ ]:
import warnings
from pathlib import Path
from tqdm import tqdm
import os
import gc
import time
import numpy as np
import soundfile as sf
from scipy import signal
import torch
from clear_memory import clear_memory

warnings.filterwarnings('ignore')

PyTorch版本: 2.8.0+cu128
CUDA可用: True
MPS可用: False


In [ ]:
input_dir = Path('../ad_detection/data/raw/Pitt')
output_dir = Path('../ad_detection/data/denoised/Pitt-FRCRN_SE')

control_files = list((input_dir / 'Control').glob('*.wav'))
dementia_files = list((input_dir / 'Dementia').glob('*.wav'))

Control组文件数: 242
Dementia组文件数: 309


## Load Model

In [ ]:
from clearvoice import ClearVoice

model_name = 'FRCRN_SE_16K'
target_sr = 16000  # 目标采样率，必须与模型匹配

myClearVoice = ClearVoice(
    task='speech_enhancement',
    model_names=[model_name]
)

加载模型: FRCRN_SE_16K...
✓ 模型加载完成


## Denoise Function

In [5]:
def denoise_audio(audio_path, model, target_sr=16000):
    """
    使用 ClearerVoice 进行语音降噪和增强
    
    Args:
        audio_path: 输入音频文件路径
        model: ClearVoice 模型实例
        target_sr: 目标采样率（16000 或 48000，取决于模型）
    
    Returns:
        denoised_audio: 降噪后的音频 numpy array
        sr: 采样率
    """
    # 加载音频
    audio, sr = sf.read(str(audio_path))
    
    # 如果是多声道，先转为单声道（在重采样之前）
    if len(audio.shape) == 2:
        # audio 形状是 (samples, channels)
        audio = np.mean(audio, axis=1)
    
    # 重采样到目标采样率（使用 scipy，更稳定）
    if sr != target_sr:
        # 计算目标样本数
        num_samples = int(len(audio) * target_sr / sr)
        audio = signal.resample(audio, num_samples)
    
    # 确保是 float32 类型
    audio = audio.astype(np.float32)
    
    # 转换为 [batch, length] 格式
    audio = np.reshape(audio, [1, audio.shape[0]])
    
    # 应用 ClearVoice 降噪
    # 使用 torch.no_grad() 禁用梯度计算，节省显存
    with torch.no_grad():
        # online_write=False 表示返回 numpy 数组而不是直接写入文件
        output_wav = model(audio, online_write=False)
    
    # output_wav 形状: [batch, length]
    return output_wav[0, :], target_sr

In [ ]:
def batch_denoise(files, output_subdir, model, target_sr, group_name):
    """
    批量降噪处理（每个文件前后都清理显存）
    
    Args:
        files: 待处理的音频文件列表
        output_subdir: 输出子目录
        model: ClearVoice 模型实例
        target_sr: 目标采样率
        group_name: 组名（用于显示进度）
    """
    # 创建输出目录
    output_subdir.mkdir(parents=True, exist_ok=True)
    
    success_count = 0
    skip_count = 0
    fail_count = 0
    
    for audio_file in tqdm(files, desc=f"Processing {group_name}"):
        output_file = output_subdir / audio_file.name
        
        # 跳过已处理的文件
        if output_file.exists():
            skip_count += 1
            continue
        
        try:
            # ⚡ 处理前清理显存
            clear_memory()
            
            # 降噪
            denoised_audio, sr = denoise_audio(audio_file, model, target_sr)
            
            # 保存（16位整数格式）
            sf.write(str(output_file), denoised_audio, sr, subtype='PCM_16')
            success_count += 1
            
            # ⚡ 处理后立即清理显存
            del denoised_audio  # 删除大数组
            clear_memory()
            
        except Exception as e:
            fail_count += 1
            print(f"\nFailed: {audio_file.name}: {e}")
            # ⚡ 失败后也要清理显存
            clear_memory()
    
    # 打印统计信息
    print(f"\n{group_name} 处理完成:")
    print(f"成功: {success_count}")
    print(f"跳过: {skip_count}")
    print(f"失败: {fail_count}")
    print(f"总计: {len(files)}")

## Execute denoise function

In [ ]:
clear_memory()

batch_denoise(
    dementia_files,
    output_dir / 'Dementia',
    myClearVoice,
    target_sr=target_sr,
    group_name='Dementia'
)

clear_memory()

batch_denoise(
    control_files,
    output_dir / 'Control',
    myClearVoice,
    target_sr=target_sr,
    group_name='Control'
)

初始显存状态: CUDA - 已分配: 0.05 GB, 已保留: 0.06 GB
清理后显存: CUDA - 已分配: 0.05 GB, 已保留: 0.06 GB

开始处理 Dementia 组


降噪处理 Dementia:   2%|▏         | 6/309 [00:13<12:18,  2.44s/it]


✗ 处理失败: 003-0.wav: Cannot interpret '3791426' as a data type


降噪处理 Dementia:   8%|▊         | 25/309 [00:47<09:04,  1.92s/it]


✗ 处理失败: 122-1.wav: Cannot interpret '2275034' as a data type


...

降噪处理 Dementia:  83%|████████▎ | 257/309 [07:17<01:11,  1.37s/it]


✗ 处理失败: 539-0.wav: Cannot interpret '1964354' as a data type


降噪处理 Dementia: 100%|██████████| 309/309 [08:32<00:00,  1.66s/it]



Dementia 处理完成:
  ✓ 成功: 280
  ⊘ 跳过: 0
  ✗ 失败: 29
  Σ 总计: 309

组间清理后显存: CUDA - 已分配: 0.06 GB, 已保留: 0.08 GB

开始处理 Control 组


降噪处理 Control:  20%|█▉        | 48/242 [01:07<03:58,  1.23s/it]